# Bab 3 — Introduction to Neural Prediction: Forward Propagation

**Acuan:** Trask, *Grokking Deep Learning*, Bab 3, hlm. 21–46. Implementasi di sini ditulis mandiri dengan contoh angka kecil yang sejalan dengan urutan konsep buku. Semua output adalah prediksi **sebelum training**.

**Tujuan belajar:** menghitung prediksi dari satu input, banyak input, banyak output, hingga dua layer; memeriksa shape matriks; dan memahami peran bobot serta aktivasi.

## 1. Satu input → satu output

Neuron paling sederhana tanpa bias maupun aktivasi menghitung $\hat y=xw$. Bobot $w$ menyatakan sensitivitas prediksi terhadap input. Mengalikan dengan bobot bukan 'berpikir'; tanpa proses belajar, nilainya hanya pemetaan yang kita tentukan. Nilai output juga belum tentu probabilitas walaupun persoalan aslinya kelas biner.

In [1]:
import numpy as np

x = 8.5        # fitur sintetis: rata-rata 'toes' tim
w = 0.12       # bobot awal, belum dilatih
y_hat = x * w
print(f'Prediksi = {x} × {w} = {y_hat:.3f}')
assert np.isclose(y_hat, 1.02)

Prediksi = 8.5 × 0.12 = 1.020


## 2. Banyak input → satu output

Untuk $d$ fitur, $\hat y=\sum_{j=1}^{d} x_jw_j=x^\top w$. Ini disebut *weighted sum* atau dot product. Setiap fitur harus dipasangkan dengan bobot yang tepat; urutan dan skala fitur berpengaruh. Jika dua fitur sangat berbeda skala, besar bobot mentah tidak bisa langsung dibandingkan sebagai 'kepentingan'.

In [2]:
features = np.array([8.5, 0.65, 1.2])  # toes, win/loss record, fans (toy)
weights = np.array([0.10, 0.20, -0.10])
manual = sum(float(feature * weight) for feature, weight in zip(features, weights))
vectorized = float(features @ weights)
print('Hitung manual:', round(manual, 3), 'NumPy dot:', round(vectorized, 3))
assert np.isclose(manual, vectorized) and features.shape == weights.shape

Hitung manual: 0.86 NumPy dot: 0.86


## 3. Satu input → banyak output

Jika satu angka input perlu menghasilkan $k$ prediksi, gunakan vektor bobot $(w_1,\ldots,w_k)$ dan $\hat{\boldsymbol y}=x\boldsymbol w$. Tanpa hidden layer, masing-masing output terpisah: mengubah bobot output pertama tidak langsung mengubah output lain. Output dapat mewakili besaran berbeda, sehingga masing-masing memerlukan target dan skala/loss yang tepat saat training.

In [3]:
record = 0.65
output_weights = np.array([0.30, 0.20, 0.90])
out = record * output_weights
for label, prediction in zip(['win', 'sad', 'hurt'], out):
    print(label, round(float(prediction), 3))
assert out.shape == (3,)

win 0.195
sad 0.13
hurt 0.585


## 4. Banyak input → banyak output

Satukan bobot dalam matriks $W\in\mathbb{R}^{d\times k}$. Untuk satu contoh $x\in\mathbb{R}^{d}$, hasil $xW\in\mathbb{R}^{k}$. Untuk $n$ contoh sekaligus, $X\in\mathbb{R}^{n\times d}$ memberi $XW\in\mathbb{R}^{n\times k}$. Baris matriks input = contoh; kolom = fitur. Pemeriksaan dimensi adalah pertahanan pertama terhadap bug perhitungan yang tampak masuk akal namun salah.

In [4]:
W = np.array([[.10, .05], [.20, -.10], [-.10, .30]])  # 3 fitur × 2 output
X = np.array([[8.5, .65, 1.2], [9.5, .80, 1.3]])
Y_hat = X @ W
print('X:', X.shape, 'W:', W.shape, 'prediksi:', Y_hat.shape)
print(Y_hat.round(3))
assert Y_hat.shape == (2, 2)
assert np.allclose(Y_hat[0], features @ W)

X: (2, 3) W: (3, 2) prediksi: (2, 2)
[[0.86  0.72 ]
 [0.98  0.785]]


## 5. Prediksi di atas prediksi: hidden layer

Dengan dua matriks, $H=\phi(XW_1+b_1)$ dan $\hat Y=HW_2+b_2$. $H$ adalah representasi intermediate, bukan target yang diamati. Tanpa aktivasi nonlinier, $(XW_1)W_2=X(W_1W_2)$ sehingga dua layer linear setara satu layer linear. ReLU $\phi(z)=\max(0,z)$ memperkenalkan nonlinieritas, tetapi juga dapat mematikan unit jika selalu menerima nilai negatif. Di bab ini bobot masih belum dioptimalkan.

In [5]:
rng = np.random.default_rng(3)
W1 = rng.normal(0, .2, size=(3, 4))
W2 = rng.normal(0, .2, size=(4, 2))
hidden = np.maximum(0, X @ W1)
deep_prediction = hidden @ W2
linear_equivalent = X @ (W1 @ W2)
print('hidden:', hidden.shape, 'output:', deep_prediction.shape)
print('Jarak output nonlinear vs linear:', round(float(np.linalg.norm(deep_prediction - linear_equivalent)), 5))
assert hidden.shape == (2, 4) and deep_prediction.shape == (2, 2)

hidden: (2, 4) output: (2, 2)
Jarak output nonlinear vs linear: 0.95813


### Ringkasan dan refleksi

- Forward propagation mengalirkan input melalui operasi berbobot menuju prediksi, tanpa mengubah bobot.
- Dot product dan perkalian matriks adalah bentuk ringkas dari weighted sums.
- Dimensi $n\times d$, $d\times h$, dan $h\times k$ harus konsisten.
- Banyak layer linear saja tidak menambah jenis fungsi yang bisa direpresentasikan; aktivasi nonlinier penting untuk pola kompleks.
- Output mentah di contoh ini bukan probabilitas terkalibrasi dan belum mengukur akurasi pada data baru.